In [1]:
%load_ext line_profiler

In [8]:
import numpy as np

In [11]:
def my_slow_function():
    a = np.arange(0, 1000*1000)
    a = a**2
    #for i in range (1000*1000):
    #    a.append(i**2)
    b = np.sum(a)
    return b


In [12]:
%lprun -f my_slow_function my_slow_function()

Timer unit: 1e-07 s

Total time: 0.0059114 s
File: C:\Users\sfreeman\AppData\Local\Temp\ipykernel_23880\466546008.py
Function: my_slow_function at line 1

Line #      Hits         Time  Per Hit   % Time  Line Contents
     1                                           def my_slow_function():
     2         1      19790.0  19790.0     33.5      a = np.arange(0, 1000*1000)
     3         1      33124.0  33124.0     56.0      a = a**2
     4                                               #for i in range (1000*1000):
     5                                               #    a.append(i**2)
     6         1       6197.0   6197.0     10.5      b = np.sum(a)
     7         1          3.0      3.0      0.0      return b

In [13]:
import geopandas as gpd
from shapely.geometry import Point
import random

# Simulate 5,000 random points 
random.seed(42)
points = [Point(random.uniform(-87, -84), random.uniform(34, 36)) for _ in range(5000)]
sightings = gpd.GeoDataFrame(geometry=points, crs="EPSG:4326")

# Load country/region polygons
counties = gpd.read_file(
    "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
)
counties = counties[counties["CONTINENT"] == "North America"][["NAME", "geometry"]].to_crs("EPSG:4326")

def label_points(sightings, counties):
    results = []
    for i, point in enumerate(sightings.geometry):       
        for j, row in counties.iterrows():               
            if row.geometry.contains(point):
                results.append(row["NAME"])
                break
        else:
            results.append(None)
    return results



In [14]:
%lprun -f label_points label_points(sightings, counties)

Timer unit: 1e-07 s

Total time: 2.52309 s
File: C:\Users\sfreeman\AppData\Local\Temp\ipykernel_23880\668962465.py
Function: label_points at line 16

Line #      Hits         Time  Per Hit   % Time  Line Contents
    16                                           def label_points(sightings, counties):
    17         1          7.0      7.0      0.0      results = []
    18      5001     311317.0     62.3      1.2      for i, point in enumerate(sightings.geometry):       
    19     10000   18641601.0   1864.2     73.9          for j, row in counties.iterrows():               
    20     10000    5628025.0    562.8     22.3              if row.geometry.contains(point):
    21      5000     586402.0    117.3      2.3                  results.append(row["NAME"])
    22      5000      63582.0     12.7      0.3                  break
    23                                                   else:
    24                                                       results.append(None)
    25         1

In [ ]:
sightings["county"] = label_points(sightings, counties)
print(sightings["county"].value_counts())